# Attari et al. (2006): what do the reported standard deviations imply?

**Source:** Attari, A., F. Yadollah Moghaddam, A. Hasanzadeh, M. Soltani, and M. Mahmoodi. "Comparison of efficacy of fluoxetine with nortriptyline in treatment of major depression in children and adolescents: a double-blind study." *Journal of Research in Medical Sciences* 11, no. 1 (2006): 24–30. The PDF is in this folder: `Attari et al Fluoxetine vs Nortriptyline.pdf`.

## The study in one paragraph

Forty children and adolescents (aged 7–16) with major depression were randomly assigned to fluoxetine or nortriptyline, 20 in each arm, for 8 weeks. Depression was measured with the Children's Depression Inventory (CDI), which runs from 0 to 54; higher scores mean more severe depression. For each arm the paper reports the mean and standard deviation (SD) of the CDI score before and after treatment, and the mean and SD of each child's change:

| Arm | Before: mean (SD) | After: mean (SD) | Change: mean (SD) |
|---|---|---|---|
| Fluoxetine | 28.9 (8.4) | 17.9 (8.1) | −10.9 (2.6) |
| Nortriptyline | 28.4 (8.7) | 25.8 (8.5) | −2.6 (0.8) |

## Why the change SDs are interesting

The three SDs in a row are not free to take any values. They are tied together by the correlation *r* between a child's score before and after treatment:

$$SD_{change}^2 = SD_{pre}^2 + SD_{post}^2 - 2\,r\,SD_{pre}\,SD_{post}$$

Scores of about 8–9 points SD before and after, combined with change SDs of only 2.6 and 0.8, are possible only if *r* is very high: about **0.951** for fluoxetine and **0.996** for nortriptyline. In other words, almost every child would have to keep nearly the same rank in the group from start to finish.

The change SD also drives the **within-group standardized mean difference (SMD)**, defined as mean change / SD of change. The reported numbers give SMD ≈ −4.19 for fluoxetine and −3.25 for nortriptyline. Both are extremely large, and both are very sensitive to *r*.

The two explorers below let you vary these inputs and watch the consequences.

## Before you start

- This is **arithmetic on published summary statistics**. It does not re-analyse patient data, and the correlation does not *cause* anything. It only links the reported numbers to one another.
- The SMD here is a **within-group** change (one arm, before vs. after). It is **not** the between-group effect of fluoxetine vs. nortriptyline, so don't compare the two panels to read off a treatment effect.
- For fluoxetine, 17.9 − 28.9 = −11.0, but the paper reports a change of −10.9, presumably because of rounding. The notebook uses the reported −10.9.

## How to run it

You need Python with `numpy`, `matplotlib` and `ipywidgets` (`pip install numpy matplotlib ipywidgets`). Open the notebook in Jupyter, JupyterLab or VS Code and run all cells from top to bottom. The sliders need a running Python kernel. On GitHub or other static viewers you will see only the text and code.

In [ ]:
# Setup: study data, formulas and helpers shared by both explorers below.
# Source: Attari et al. (2006), Journal of Research in Medical Sciences 11(1): 24-30
# (full citation at the top of the notebook).

import io
from dataclasses import dataclass

import numpy as np
import matplotlib.pyplot as plt
import ipywidgets as widgets
from IPython.display import display


# -----------------------------------------------------------------------------
# Study data: CDI scores per treatment arm (n = 20 each), from Table 2 and the
# Results section of the paper.
# -----------------------------------------------------------------------------
@dataclass(frozen=True)
class Arm:
    name: str
    mean_pre: float
    mean_post: float
    mean_change: float  # as reported (post - pre); see the note below
    sd_pre: float
    sd_post: float
    sd_change: float    # reported SD of the individual change scores

    @property
    def r_reported(self):
        """Pre-post correlation implied by the three reported SDs."""
        return implied_r(self.sd_pre, self.sd_post, self.sd_change)

    @property
    def smd_reported(self):
        """Within-group SMD implied by the reported numbers."""
        return self.mean_change / self.sd_change


ARMS = [
    Arm("Fluoxetine", mean_pre=28.9, mean_post=17.9, mean_change=-10.9,
        sd_pre=8.4, sd_post=8.1, sd_change=2.6),
    Arm("Nortriptyline", mean_pre=28.4, mean_post=25.8, mean_change=-2.6,
        sd_pre=8.7, sd_post=8.5, sd_change=0.8),
]
# Note: for Fluoxetine, 17.9 - 28.9 = -11.0, but the paper reports a change of -10.9,
# presumably because the published means are rounded. We use the reported change.


# -----------------------------------------------------------------------------
# Formulas
# -----------------------------------------------------------------------------
def change_sd(sd_pre, sd_post, r):
    """SD of the change score (post - pre), given the pre-post correlation r:

        Var(post - pre) = SD_pre^2 + SD_post^2 - 2 r SD_pre SD_post

    Returns NaN where the SD is (numerically) zero, so dividing by it is always safe."""
    var = sd_pre**2 + sd_post**2 - 2.0 * np.asarray(r, dtype=float) * sd_pre * sd_post
    sd = np.sqrt(np.clip(var, 0.0, None))
    return np.where(sd > 1e-6, sd, np.nan)


def standardized_change(mean_change, sd_pre, sd_post, r):
    """Within-group standardized mean difference: SMD = mean change / SD of change.
    Negative means the CDI score fell, i.e. depression improved."""
    return mean_change / change_sd(sd_pre, sd_post, r)


def implied_r(sd_pre, sd_post, sd_change):
    """The correlation r that makes change_sd(sd_pre, sd_post, r) equal sd_change.
    A value outside [-1, 1] means no real correlation can produce that change SD."""
    return (sd_pre**2 + sd_post**2 - sd_change**2) / (2.0 * sd_pre * sd_post)


# -----------------------------------------------------------------------------
# Plotting and display helpers
# -----------------------------------------------------------------------------
COLORS = {
    "curve": "#2a78d6",     # curve for the current slider values
    "paper": "#0b0b0b",     # fixed point from the paper's reported numbers
    "required": "#eb6834",  # highlight ring (explorer 1)
    "guide": "#898781",     # probe line / distance-to-curve line
    "grid": "#e1e0d9",
    "axis": "#c3c2b7",
    "muted": "#52514e",     # secondary text
    "surface": "#fcfcfb",   # plot background
    "good": "#006300",
    "bad": "#b3261e",
}


def style_axes(ax, title, xlabel, ylabel):
    ax.set_facecolor(COLORS["surface"])
    ax.set_title(title, fontsize=12, loc="left")
    ax.set_xlabel(xlabel)
    ax.set_ylabel(ylabel)
    ax.grid(True, color=COLORS["grid"], lw=0.8)
    ax.set_axisbelow(True)
    for side in ("top", "right"):
        ax.spines[side].set_visible(False)
    for side in ("left", "bottom"):
        ax.spines[side].set_color(COLORS["axis"])
    ax.tick_params(colors=COLORS["muted"])


def fig_to_png(fig):
    """Render a figure to PNG bytes and close it, so Jupyter does not also display it."""
    buf = io.BytesIO()
    fig.savefig(buf, format="png", facecolor=fig.get_facecolor())
    plt.close(fig)
    return buf.getvalue()


def html_table(rows):
    """Two-column (label, value) HTML table for the numeric summary under each plot."""
    body = "".join(f"<tr><td style='padding:2px 12px 2px 0;color:{COLORS['muted']}'>{k}</td>"
                   f"<td style='padding:2px 0;font-variant-numeric:tabular-nums'>{v}</td></tr>"
                   for k, v in rows)
    return f"<table style='font-size:13px;border-collapse:collapse'>{body}</table>"


def note(text):
    """Grey inline note, e.g. the paper's value next to the current one."""
    return f"&nbsp;<span style='color:{COLORS['guide']}'>({text})</span>"


def slider(description, value, lo, hi, step, fmt, label_width="70px", width="95%"):
    return widgets.FloatSlider(value=value, min=lo, max=hi, step=step, description=description,
                               readout_format=fmt, continuous_update=True,
                               style={"description_width": label_width},
                               layout=widgets.Layout(width=width))


# -----------------------------------------------------------------------------
# Widget layout shared by both explorers: one column per arm, side by side, plus
# a shared axis slider and a "Reset" button.
# -----------------------------------------------------------------------------
class ArmPanel:
    """One column: a heading, the arm's sliders, its plot and a summary table.

    `draw(ax, **values)` and `summarize(**values)` receive the current slider values,
    keyed like `sliders`. The values the sliders start at are what "Reset" restores."""

    def __init__(self, arm, sliders, draw, summarize, figsize):
        self.sliders, self.draw, self.summarize, self.figsize = sliders, draw, summarize, figsize
        self.defaults = {k: s.value for k, s in sliders.items()}
        self._paused = False
        # The plot is a PNG swapped in place on every change. This avoids plots piling
        # up in front ends (e.g. VS Code) where clearing an Output widget is unreliable.
        self.image = widgets.Image(format="png", layout=widgets.Layout(width="100%"))
        self.summary = widgets.HTML()
        for s in sliders.values():
            s.observe(self.update, names="value")
        self.widget = widgets.VBox(
            [widgets.HTML(f"<h3 style='margin:4px 0'>{arm.name}</h3>"), *sliders.values(),
             self.image, self.summary],
            layout=widgets.Layout(width="50%", padding="0 10px"))
        self.update()

    def update(self, *_):
        if self._paused:
            return
        values = {k: s.value for k, s in self.sliders.items()}
        fig, ax = plt.subplots(figsize=self.figsize, dpi=100)
        fig.patch.set_facecolor(COLORS["surface"])
        self.draw(ax, **values)
        fig.tight_layout()
        self.image.value = fig_to_png(fig)
        self.summary.value = self.summarize(**values)

    def reset(self):
        self._paused = True  # set every slider first, then redraw once
        for k, s in self.sliders.items():
            s.value = self.defaults[k]
        self._paused = False
        self.update()


def assemble(panels, shared_slider, reset_tooltip):
    """Lay the panels out side by side under a Reset button and a shared axis slider."""
    shared_default = shared_slider.value

    def redraw_all(*_):
        for p in panels:
            p.update()

    def reset_all(_):
        shared_slider.unobserve(redraw_all, names="value")  # avoid a redundant redraw
        shared_slider.value = shared_default
        shared_slider.observe(redraw_all, names="value")
        for p in panels:
            p.reset()

    shared_slider.observe(redraw_all, names="value")
    reset_btn = widgets.Button(description="Reset", icon="undo", button_style="primary",
                               tooltip=reset_tooltip)
    reset_btn.on_click(reset_all)
    return widgets.VBox([
        widgets.HBox([reset_btn, shared_slider],
                     layout=widgets.Layout(align_items="center", gap="16px")),
        widgets.HBox([p.widget for p in panels]),
    ])

## Explorer 1: how much does the SMD depend on the correlation?

For each arm, the blue curve shows the standardized mean difference (SMD) for every possible pre-post correlation *r*, from −1 to 1, given the current SD sliders:

$$SMD(r) = \frac{\text{mean change}}{\sqrt{SD_{pre}^2 + SD_{post}^2 - 2\,r\,SD_{pre}\,SD_{post}}}$$

**On the plot**

- **Black diamond:** the paper's own numbers, meaning the *r* implied by its three SDs and the SMD implied by its mean change and change SD. It never moves.
- **Orange ring:** the *r* the current SD sliders would need in order to reproduce the paper's change SD. With the original SDs it sits on the diamond. It disappears when no valid *r* (between −1 and 1) can do it.
- **Grey dotted line and dot:** the probe. Pick any *r* to read off the change SD and SMD it implies. The values are listed in the table under each plot.

**Sliders**

- **SD pre / SD post:** try other SDs than the published ones.
- **Probe r:** the correlation to read off.
- **Shared y-axis floor:** the bottom of both y-axes. When SD pre ≈ SD post, SMD plunges toward −∞ as *r* → 1, so you may need to lower it.
- **Reset:** restore the paper's values in both panels.

Things to try: watch how steep the curve is near the paper's *r*, and how much SMD changes between *r* = 0.5 and *r* = 0.95.

In [ ]:
# Explorer 1: standardized mean difference (SMD) as a function of the pre-post correlation r.

R_GRID = np.linspace(-1.0, 1.0, 4001)  # r values the curve is drawn over
Y_TOP = 0.5
Y_FLOOR = -15.0                        # default bottom of the y-axis, shared by both panels


def draw_smd_vs_r(ax, arm, sd_pre, sd_post, r_probe, y_floor=Y_FLOOR):
    curve = standardized_change(arm.mean_change, sd_pre, sd_post, R_GRID)
    smd_probe = float(standardized_change(arm.mean_change, sd_pre, sd_post, r_probe))
    r_required = implied_r(sd_pre, sd_post, arm.sd_change)

    ax.axhline(0.0, color=COLORS["axis"], lw=1.2, zorder=1)  # SMD = 0: no change
    # Where the change SD is zero the curve is NaN, and matplotlib leaves a gap.
    ax.plot(R_GRID, curve, color=COLORS["curve"], lw=2.2, zorder=3,
            label=f"Current curve (SD pre {sd_pre:.1f}, SD post {sd_post:.1f})")

    ax.axvline(r_probe, color=COLORS["guide"], lw=1, ls=":", zorder=2)
    if np.isfinite(smd_probe) and y_floor <= smd_probe <= Y_TOP:
        ax.plot(r_probe, smd_probe, "o", ms=7, color=COLORS["guide"], mec="white", mew=1.5,
                zorder=4, label=f"Probe r = {r_probe:.3f}")

    # The r at which the *current* SDs reproduce the paper's change SD. Its height is
    # always the paper's SMD. Drawn as a hollow ring so it stays visible when it sits on
    # top of the paper's point (as it does with the original SDs).
    if -1.0 <= r_required <= 1.0:
        ax.plot(r_required, arm.smd_reported, "o", ms=16, mfc="none", mec=COLORS["required"],
                mew=2.2, zorder=7, label="r needed to match the reported change SD")

    # Fixed point from the paper's numbers; it does not move with the sliders.
    ax.plot(arm.r_reported, arm.smd_reported, "D", ms=10, color=COLORS["paper"], mec="white",
            mew=1.5, zorder=6, label="Attari et al. (implied r, reported SMD)")
    ax.annotate(f"r = {arm.r_reported:.3f}\nSMD = {arm.smd_reported:.2f}",
                (arm.r_reported, arm.smd_reported), xytext=(-12, 0), textcoords="offset points",
                ha="right", va="center", fontsize=9, color=COLORS["muted"])

    # When SD pre ~ SD post the curve plunges as r -> 1; say so if it leaves the plot.
    finite = curve[np.isfinite(curve)]
    if finite.size and finite.min() < y_floor:
        ax.text(0.02, 0.03, f"Curve continues below {y_floor:.0f} (lowest value {finite.min():.1f})",
                transform=ax.transAxes, fontsize=8, color=COLORS["muted"])

    ax.set_xlim(-1.02, 1.02)
    ax.set_ylim(y_floor, Y_TOP)
    style_axes(ax, f"{arm.name}: within-group standardized change",
               "Pre-post correlation (r)", "Standardized mean difference (SMD)")
    ax.legend(loc="lower left", bbox_to_anchor=(0.0, 0.08), fontsize=8.5, frameon=False)


def summarize_smd_vs_r(arm, sd_pre, sd_post, r_probe):
    ssmd_probe = float(change_sd(sd_pre, sd_post, r_probe))
    smd_probe = arm.mean_change / ssmd_probe
    r_required = implied_r(sd_pre, sd_post, arm.sd_change)

    if -1.0 <= r_required <= 1.0:
        required = f"{r_required:.4f} <span style='color:{COLORS['good']}'>(possible)</span>"
    else:
        required = (f"{r_required:.4f} <span style='color:{COLORS['bad']}'>(outside [-1, 1]: "
                    f"no correlation gives a change SD of {arm.sd_change} with these SDs)</span>")

    def fmt(x):
        return f"{x:.3f}" if np.isfinite(x) else "undefined (change SD is 0)"

    return html_table([
        ("Current SD pre / SD post", f"{sd_pre:.1f} / {sd_post:.1f}"),
        ("Mean change (reported)",
         f"{arm.mean_change:.1f}" + note(f"post - pre = {arm.mean_post - arm.mean_pre:.1f}")),
        ("Reported change SD", f"{arm.sd_change}"),
        ("r implied by the paper's SDs", f"{arm.r_reported:.4f}"),
        ("SMD implied by the paper's numbers", f"{arm.smd_reported:.3f}"),
        ("r needed to match the reported change SD (current SDs)", required),
        (f"At probe r = {r_probe:.3f}: change SD", fmt(ssmd_probe)),
        (f"At probe r = {r_probe:.3f}: SMD", fmt(smd_probe)),
    ])


def smd_vs_r_panel(arm, floor_slider):
    sliders = {
        "sd_pre": slider("SD pre", arm.sd_pre, 0.5, 20.0, 0.1, ".1f"),
        "sd_post": slider("SD post", arm.sd_post, 0.5, 20.0, 0.1, ".1f"),
        "r_probe": slider("Probe r", arm.r_reported, -1.0, 1.0, 0.001, ".3f"),
    }
    return ArmPanel(arm, sliders,
                    draw=lambda ax, **v: draw_smd_vs_r(ax, arm, **v, y_floor=floor_slider.value),
                    summarize=lambda **v: summarize_smd_vs_r(arm, **v),
                    figsize=(6.4, 4.6))


y_floor = slider("Shared y-axis floor", Y_FLOOR, -60.0, -2.0, 0.5, ".1f",
                 label_width="140px", width="420px")
display(assemble([smd_vs_r_panel(arm, y_floor) for arm in ARMS], y_floor,
                 reset_tooltip="Restore the paper's values in both panels"))

## Explorer 2: which (SD pre, SD post) pairs fit a chosen *r* and SMD?

This works the other way round. Keep each arm's reported mean change fixed, then choose a correlation *r* and a within-group SMD. That fixes the change SD at |mean change| / |SMD|, and every pair of SDs on the blue curve is consistent with those choices:

$$SD_{pre}^2 + SD_{post}^2 - 2\,r\,SD_{pre}\,SD_{post} = \left(\frac{\text{mean change}}{SMD}\right)^2$$

**On the plot**

- **Blue curve:** all SD pairs compatible with the chosen *r* and SMD.
- **Black diamond:** the SDs the paper reports. If the curve passes through it, the chosen *r* and SMD are consistent with the paper. The table reports "Compatible" when the SMD those SDs give at your *r* is within 0.01 of your SMD.
- **Grey dotted line:** the shortest distance from the diamond to the curve.
- **Dashed diagonal:** SD pre = SD post, for reference.

**Sliders**

- **r** and **SMD (within):** start at the values implied by the paper. The *r* slider moves in very fine steps because near *r* = 0.996 even a step of 0.001 changes nortriptyline's SMD noticeably.
- **Shared axis max:** zoom both plots out when the curve runs off the axes.
- **Reset:** restore the paper's values in both panels.

Things to try: set *r* to a more typical test-retest value such as 0.5–0.7 and keep the paper's SMD. How far is the diamond from the curve? Or keep *r* at 0.5 and find the SMD that brings the curve through the diamond.

In [ ]:
# Explorer 2: which (SD pre, SD post) pairs fit a chosen r and SMD?

AXIS_MAX = 15.0  # default upper limit of both axes, shared by both panels
SMD_TOLERANCE = 0.01  # the paper's SDs count as "compatible" if they give an SMD this close


def feasible_sd_pairs(sd_change, r, n=4000):
    """Points (x, y) = (SD pre, SD post) with x, y > 0 on the curve x^2 + y^2 - 2rxy = sd_change^2.

    The curve is an ellipse tilted 45 degrees. Solving for y directly gives two branches,
    y = r x +/- sqrt(sd_change^2 - (1 - r^2) x^2), which are awkward to join. Instead we
    rotate the axes, u = (x + y)/sqrt2 and v = (y - x)/sqrt2, which turns the curve into
    (1 - r) u^2 + (1 + r) v^2 = sd_change^2, and trace it with an angle t.
    Returns a list of (x, y) arrays, one per unbroken stretch in the positive quadrant."""
    t = np.linspace(0, 2 * np.pi, n)
    u = sd_change * np.cos(t) / np.sqrt(1 - r)
    v = sd_change * np.sin(t) / np.sqrt(1 + r)
    x, y = (u - v) / np.sqrt(2), (u + v) / np.sqrt(2)

    idx = np.flatnonzero((x > 0) & (y > 0))
    if idx.size == 0:
        return []
    runs = np.split(idx, np.flatnonzero(np.diff(idx) > 1) + 1)
    # t = 0 and t = 2*pi are the same point, so a stretch crossing it is one piece.
    if len(runs) > 1 and runs[0][0] == 0 and runs[-1][-1] == n - 1:
        runs = [np.concatenate([runs[-1], runs[0]])] + runs[1:-1]
    return [(x[run], y[run]) for run in runs if run.size > 1]


def nearest_on_curve(segments, x0, y0):
    """(x, y, distance) of the drawn curve point closest to (x0, y0), or None if no curve."""
    if not segments:
        return None
    xs = np.concatenate([s[0] for s in segments])
    ys = np.concatenate([s[1] for s in segments])
    k = np.argmin(np.hypot(xs - x0, ys - y0))
    return xs[k], ys[k], float(np.hypot(xs[k] - x0, ys[k] - y0))


def draw_feasible(ax, arm, r, smd, axis_max=AXIS_MAX):
    segments = feasible_sd_pairs(abs(arm.mean_change / smd), r)

    ax.plot([0, axis_max], [0, axis_max], color=COLORS["axis"], lw=1, ls="--", zorder=1,
            label="SD pre = SD post")
    for i, (x, y) in enumerate(segments):
        ax.plot(x, y, color=COLORS["curve"], lw=2.2, zorder=3,
                label=f"Feasible SDs (r = {r:.4f}, SMD = {smd:.3f})" if i == 0 else None)

    # Dotted line from the paper's SDs to the nearest point on the curve (hidden if ~on it).
    near = nearest_on_curve(segments, arm.sd_pre, arm.sd_post)
    if near and near[2] > 0.05:
        ax.plot([arm.sd_pre, near[0]], [arm.sd_post, near[1]], color=COLORS["guide"],
                lw=1.2, ls=":", zorder=2, label=f"Distance to curve ({near[2]:.2f})")

    ax.plot(arm.sd_pre, arm.sd_post, "D", ms=10, color=COLORS["paper"], mec="white", mew=1.5,
            zorder=5, label="Attari et al. reported SDs")
    ax.annotate(f"({arm.sd_pre}, {arm.sd_post})", (arm.sd_pre, arm.sd_post), xytext=(10, -12),
                textcoords="offset points", fontsize=9, color=COLORS["muted"])

    if any(x.max() > axis_max or y.max() > axis_max for x, y in segments):
        ax.text(0.98, 0.02, "Curve extends beyond the axes", transform=ax.transAxes,
                ha="right", fontsize=8, color=COLORS["muted"])
    if not segments:
        ax.text(0.5, 0.5, "No positive SD pair is feasible", transform=ax.transAxes,
                ha="center", color=COLORS["bad"])

    ax.set_xlim(0, axis_max)
    ax.set_ylim(0, axis_max)
    ax.set_aspect("equal")
    style_axes(ax, f"{arm.name}: feasible (SD pre, SD post)", "SD pre", "SD post")
    ax.legend(loc="upper left", fontsize=8.5, frameon=False)


def summarize_feasible(arm, r, smd):
    sd_change = abs(arm.mean_change / smd)
    smd_paper_sds = float(standardized_change(arm.mean_change, arm.sd_pre, arm.sd_post, r))
    gap = abs(smd - smd_paper_sds) if np.isfinite(smd_paper_sds) else np.inf
    r_needed = implied_r(arm.sd_pre, arm.sd_post, sd_change)

    status = (f"<b style='color:{COLORS['good']}'>Compatible</b>" if gap < SMD_TOLERANCE else
              f"<b style='color:{COLORS['bad']}'>Not compatible</b>")
    r_needed_txt = f"{r_needed:.5f}" + ("" if -1 < r_needed < 1 else " (outside (-1, 1): impossible)")

    return html_table([
        ("Mean change (reported)", f"{arm.mean_change:.1f}"),
        ("Selected r", f"{r:.5f}" + note(f"paper implies {arm.r_reported:.5f}")),
        ("Selected SMD", f"{smd:.3f}" + note(f"paper implies {arm.smd_reported:.3f}")),
        ("Change SD = |mean change| / |SMD|", f"{sd_change:.3f}"),
        ("Paper's SD pre / SD post", f"{arm.sd_pre} / {arm.sd_post}"),
        ("SMD given by the paper's SDs at the selected r",
         f"{smd_paper_sds:.3f}" if np.isfinite(smd_paper_sds) else "undefined"),
        ("Gap |selected SMD - that SMD|", f"{gap:.3f}"),
        ("r that puts the paper's SDs on this curve", r_needed_txt),
        (f"Status (tolerance {SMD_TOLERANCE})", status),
    ])


def feasible_panel(arm, axis_slider):
    sliders = {
        # Fine r steps: near r = 0.996 a step of 0.001 moves Nortriptyline's SMD by ~0.03.
        "r": slider("r", arm.r_reported, -0.99, 0.999, 1e-5, ".5f"),
        "smd": slider("SMD (within)", arm.smd_reported, -8.0, -0.1, 0.001, ".3f"),
    }
    return ArmPanel(arm, sliders,
                    draw=lambda ax, **v: draw_feasible(ax, arm, **v, axis_max=axis_slider.value),
                    summarize=lambda **v: summarize_feasible(arm, **v),
                    figsize=(5.6, 5.6))


axis_max = slider("Shared axis max", AXIS_MAX, 5, 80, 1, ".0f", label_width="120px", width="400px")
display(assemble([feasible_panel(arm, axis_max) for arm in ARMS], axis_max,
                 reset_tooltip="Restore the r and SMD implied by the paper in both panels"))